In [9]:
from pyspark.sql import SparkSession

spark = SparkSession \
    .builder \
    .appName("TradeCorp ETL") \
    .getOrCreate()

print(spark.version);

PATH = "../data/"

4.2.0


# Q2 — Lire les 8 CSV
Lire les 8 fichiers CSV. Stocker chaque DataFrame dans une variable.

In [13]:
df_categories = spark.read.csv(f"{PATH}categories.csv", header=True, inferSchema=True);
df_customers = spark.read.csv(f"{PATH}customers.csv", header=True, inferSchema=True);
df_employees = spark.read.csv(f"{PATH}employees.csv", header=True, inferSchema=True);
df_orders_details = spark.read.csv(f"{PATH}order_details.csv", header=True, inferSchema=True);
df_orders = spark.read.csv(f"{PATH}orders.csv", header=True, inferSchema=True);
df_products = spark.read.csv(f"{PATH}products.csv", header=True, inferSchema=True);
df_shippers = spark.read.csv(f"{PATH}shippers.csv", header=True, inferSchema=True);
df_suppliers = spark.read.csv(f"{PATH}suppliers.csv", header=True, inferSchema=True);

# Q3 — Explorer le schema
Pour chaque DataFrame, afficher le schema des données. Identifier les types de colonnes inférés
automatiquement.

In [16]:
print("Categories");
df_categories.printSchema();

print("Customers");
df_customers.printSchema();

print("Employees");
df_employees.printSchema();

print("Order Details");
df_orders_details.printSchema();

print("Orders");
df_orders.printSchema();

print("Products");
df_products.printSchema();

print("Shippers");
df_shippers.printSchema();

print("Suppliers");
df_suppliers.printSchema();

Categories
root
 |-- category_id: integer (nullable = true)
 |-- category_name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- picture: string (nullable = true)

Customers
root
 |-- customer_id: string (nullable = true)
 |-- company_name: string (nullable = true)
 |-- contact_name: string (nullable = true)
 |-- contact_title: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- country: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- fax: string (nullable = true)

Employees
root
 |-- employee_id: integer (nullable = true)
 |-- last_name: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- title: string (nullable = true)
 |-- title_of_courtesy: string (nullable = true)
 |-- birth_date: date (nullable = true)
 |-- hire_date: date (nullable = true)
 |-- address: string (nullable = true)
 |-- ci

La majorité des colonnes ont le bon typage fait de maniére automatique; cependant certaines ont un mauvais type par exemple Products.quantity_per_unit qui est en string alors qu'il devrait etre en integer par logique

# Q4 — Afficher les données
Pour chaque DataFrame, afficher les 5 premières lignes. Observer la structure des données.

In [17]:
print("Categories");
df_categories.show(5);

print("Customers");
df_customers.show(5);

print("Employees");
df_employees.show(5);

print("Order Details");
df_orders_details.show(5);

print("Orders");
df_orders.show(5);

print("Products");
df_products.show(5);

print("Shippers");
df_shippers.show(5);

print("Suppliers");
df_suppliers.show(5);

Categories
+-----------+--------------+--------------------+-------+
|category_id| category_name|         description|picture|
+-----------+--------------+--------------------+-------+
|          1|     Beverages|Soft drinks, coff...|   NULL|
|          2|    Condiments|Sweet and savory ...|   NULL|
|          3|   Confections|Desserts, candies...|   NULL|
|          4|Dairy Products|             Cheeses|   NULL|
|          5|Grains/Cereals|Breads, crackers,...|   NULL|
+-----------+--------------+--------------------+-------+
only showing top 5 rows
Customers
+-----------+--------------------+------------------+--------------------+--------------------+-----------+------+-----------+-------+--------------+--------------+
|customer_id|        company_name|      contact_name|       contact_title|             address|       city|region|postal_code|country|         phone|           fax|
+-----------+--------------------+------------------+--------------------+--------------------+--------

# Q5 — Compter les lignes
Compter le nombre de lignes de chaque DataFrame avec. Créer un tableau récapitulatif.

In [19]:
tab_df_lignes = []
tab_df_lignes.append(("Categories",df_categories.count()));
tab_df_lignes.append(("Customers",df_customers.count()));
tab_df_lignes.append(("Employees",df_employees.count()));
tab_df_lignes.append(("Order Details",df_orders_details.count()));
tab_df_lignes.append(("Orders",df_orders.count()));
tab_df_lignes.append(("Products",df_products.count()));
tab_df_lignes.append(("Shippers",df_shippers.count()));
tab_df_lignes.append(("Suppliers",df_suppliers.count()));

print(tab_df_lignes);

[('Categories', 8), ('Customers', 91), ('Employees', 9), ('Order Details', 2155), ('Orders', 830), ('Products', 77), ('Shippers', 6), ('Suppliers', 29)]


# Q6 — Statistiques descriptives
Sur df_orders et df_products essayer d’obtenir les statistiques suivantes : min, max, mean, stddev.

In [23]:
df_orders.describe().show();

df_products.describe().show();


+-------+-----------------+-----------+------------------+------------------+------------------+--------------------+--------------------+---------+-----------+------------------+------------+
|summary|         order_id|customer_id|       employee_id|          ship_via|           freight|           ship_name|        ship_address|ship_city|ship_region|  ship_postal_code|ship_country|
+-------+-----------------+-----------+------------------+------------------+------------------+--------------------+--------------------+---------+-----------+------------------+------------+
|  count|              830|        830|               830|               830|               830|                 830|                 830|      830|        323|               811|         830|
|   mean|          10662.5|       NULL| 4.403614457831325|2.0072289156626506| 78.24420481927719|                NULL|                NULL|     NULL|       NULL|39975.067357512955|        NULL|
| stddev|239.7446558319914|       N

# Q7 — Lazy evaluation
Expliquer dans une cellule Markdown ce qu'est la lazy evaluation dans Spark. Quelle est la différence entre une
transformation et une action ? Donner 3 exemples de chaque.

## Lazy Evaluation dans Apache Spark

### Principe
Spark utilise la **lazy evaluation** (évaluation paresseuse) : les **transformations** ne sont **pas exécutées immédiatement**.
Elles sont **enregistrées** dans un **graphe de dépendances** (DAG) et **exécutées uniquement quand une action est appelée**.
Cela optimise les performances en :
- Évitant les calculs inutiles.
- Fusionnant les opérations (ex: `filter` + `map` en une seule passe).
- Permettant la planification optimale des tâches sur le cluster.

---

### ⚙️ Différence : Transformation vs Action

| **Transformation**                          | **Action**                              |
|--------------------------------------------|-----------------------------------------|
| **Paresseuse** : détermine le *quoi* (logique), pas le *quand*. | **Déclencheuse** : force l'exécution du DAG et renvoie un résultat. |
| Crée un nouveau **RDD/DataFrame**.         | Renvoie un **résultat concret** (scalaire, liste, fichier). |
| Exemples : `map`, `filter`, `groupBy`.      | Exemples : `collect`, `count`, `write`. |
| **Pas d'I/O** tant qu'aucune action n'est appelée. | **Provoque l'I/O** (lecture/écriture disque ou réseau). |

---

### 📌 Exemples

#### Transformations (lazy)

1. **`filter()`** : Filtre les lignes selon une condition.
```python
   df.filter(df.age > 18)  # Aucun calcul maintenant
```

2. **`select()`** : Sélectionne des colonnes.
```python
   df.select("name", "salary")
```

3. **`groupBy()`** : Regroupe les données par clé.
```python
   df.groupBy("department")
```

#### Actions (trigger)

1. **`count()`** : Compte le nombre de lignes.
```python
   df.count()  # Exécute TOUTES les transformations précédentes
```

2. **`collect()`** : Récupère TOUTES les données sur le driver (⚠️ à éviter sur gros datasets).
```python
   df.collect()
```

3. **`write.csv()`** : Écrit le résultat dans un fichier.
```python
   df.write.csv("output/")  # Déclenche le calcul et sauvegarde
```

# Q8 — Spark UI
Ouvrir http://localhost:4040 dans le navigateur. Observer les jobs exécutés. Identifier ce que représentent les
stages et tasks.

## Stages et Tasks dans Spark

**Hiérarchie** : Job → Stages → Tasks

### Stages
Ensemble de transformations exécutables **sans shuffle**. Spark crée un nouveau stage à chaque opération "wide" nécessitant un échange de données entre partitions (`groupBy`, `join`, `sort`...).

    df.filter(df.age > 18).groupBy("department").count()

- Stage 1 : `filter` (narrow, pas de shuffle)
- Stage 2 : `groupBy` + `count` (shuffle → nouveau stage)

### Tasks
Plus petite unité de travail : exécution d'un stage sur **une seule partition**. Si le DataFrame a 8 partitions, le stage lance 8 tasks en parallèle.

### Schéma

    Job (déclenché par une action)
     └── Stage 1 (narrow, pas de shuffle)
          ├── Task 1 (partition 1)
          ├── Task 2 (partition 2)
          └── Task 3 (partition 3)
     └── Stage 2 (après shuffle)
          ├── Task 1 (partition 1)
          └── Task 2 (partition 2)